# ES 335: Baselines and model ablation (train 2017-2020, validate on 2021)
Requires `corn_model_table.csv` and `corn_yield.csv` in your Drive folder (made by the data pipeline notebook).
**Do not use 2022 here.** It is reserved for the single final test.

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/es335_crop"

df     = pd.read_csv(f"{DATA_DIR}/corn_model_table.csv", dtype={"FIPS": str, "STATEFP": str})
corn_y = pd.read_csv(f"{DATA_DIR}/corn_yield.csv",       dtype={"FIPS": str})
print(df.shape, "| years:", sorted(df["Year"].unique()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(8691, 62) | years: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


## 1. Split and baselines

In [ ]:
train = df[df["Year"].between(2017, 2020)]
val   = df[df["Year"] == 2021].copy()

# Previous-year yield (2016 is used only to support 2017)
prev = corn_y.assign(Year=corn_y["Year"] + 1).rename(columns={"Yield": "prev_yield"})
val  = val.merge(prev[["FIPS", "Year", "prev_yield"]], on=["FIPS", "Year"], how="left")

# Baseline A: county mean over training years (fallback: state mean, then overall mean)
county_mean = train.groupby("FIPS")["Yield"].mean()
state_mean  = train.groupby("STATEFP")["Yield"].mean()
val["p_county_mean"] = (val["FIPS"].map(county_mean)
                        .fillna(val["STATEFP"].map(state_mean))
                        .fillna(train["Yield"].mean()))
# Baseline B: last year's yield for the same county (fallback: county mean)
val["p_prev_year"] = val["prev_yield"].fillna(val["p_county_mean"])

def scores(y, p):
    return {"RMSE": np.sqrt(mean_squared_error(y, p)),
            "MAE": mean_absolute_error(y, p),
            "R2": r2_score(y, p)}

print("val rows:", len(val),
      "| no county history:", (~val["FIPS"].isin(county_mean.index)).sum(),
      "| no prev-year value:", val["prev_yield"].isna().sum())
base_res = pd.DataFrame({"county_mean": scores(val["Yield"], val["p_county_mean"]),
                         "prev_year":   scores(val["Yield"], val["p_prev_year"])}).T.round(2)
print(base_res)

val rows: 1469 | no county history: 39 | no prev-year value: 110
              RMSE    MAE    R2
county_mean  23.83  18.77  0.61
prev_year    27.51  20.98  0.48


## 2. Models x feature sets (ablation)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

weather_cols = [c for c in df.columns if re.match(r"(tavg|tmax|tmin|precip|rh|vpd|srad|heat_days)_", c)]
soil_cols    = [c for c in df.columns if c.startswith(("nitrogen", "soc", "phh2o"))]
geo_cols     = ["lat", "lon"]
feature_sets = {
    "weather":          weather_cols,
    "weather+geo":      weather_cols + geo_cols,
    "weather+geo+soil": weather_cols + geo_cols + soil_cols,
}
print({k: len(v) for k, v in feature_sets.items()})

# Imputation and scaling live INSIDE the pipelines, so they are fit on training data only.
models = {
    "ridge": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10)),
    "rf":    lambda: make_pipeline(SimpleImputer(strategy="median"),
                                   RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                                         n_jobs=-1, random_state=42)),
    "hgb":   lambda: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42),
}

rows = []
for fs, cols in feature_sets.items():
    for name, make in models.items():
        m = make().fit(train[cols], train["Yield"])
        rows.append({"features": fs, "model": name, **scores(val["Yield"], m.predict(val[cols]))})

res = pd.DataFrame(rows)
base = scores(val["Yield"], val["p_county_mean"])["RMSE"]
res["RMSE_gain_vs_county_mean_%"] = ((base - res["RMSE"]) / base * 100).round(1)
print("county-mean baseline RMSE:", round(base, 2))
print(res.round(2).to_string(index=False))
res.to_csv(f"{DATA_DIR}/ablation_val2021.csv", index=False)

{'weather': 50, 'weather+geo': 52, 'weather+geo+soil': 58}
county-mean baseline RMSE: 23.83
        features model  RMSE   MAE   R2  RMSE_gain_vs_county_mean_%
         weather ridge 35.17 26.93 0.15                       -47.6
         weather    rf 30.78 24.36 0.35                       -29.2
         weather   hgb 30.23 23.21 0.37                       -26.9
     weather+geo ridge 36.41 28.91 0.09                       -52.8
     weather+geo    rf 27.26 20.99 0.49                       -14.4
     weather+geo   hgb 26.59 21.35 0.51                       -11.6
weather+geo+soil ridge 33.94 25.72 0.21                       -42.4
weather+geo+soil    rf 25.68 19.71 0.55                        -7.8
weather+geo+soil   hgb 25.61 20.07 0.55                        -7.5


Next steps: paste the results table into the chat, then add (1) leave-states-out validation, (2) light tuning scored on 2021 only, (3) one final test on 2022.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, ParameterGrid

best_features = feature_sets["weather+geo+soil"] # Using the most comprehensive set[cite: 3]

# --- 1. Leave-States-Out Validation (Train 2017-2020) ---
print("--- 1. Leave-States-Out Validation ---")
logo = LeaveOneGroupOut()
lso_rmse = []

# Split the training data by the STATEFP column
for train_idx, test_idx in logo.split(train[best_features], train["Yield"], train["STATEFP"]):
    X_train_cv, y_train_cv = train[best_features].iloc[train_idx], train["Yield"].iloc[train_idx]
    X_test_cv, y_test_cv   = train[best_features].iloc[test_idx],  train["Yield"].iloc[test_idx]

    # Using your existing hgb pipeline initialization[cite: 3]
    m = models["hgb"]().fit(X_train_cv, y_train_cv)
    preds = m.predict(X_test_cv)
    lso_rmse.append(scores(y_test_cv, preds)["RMSE"])

print(f"Average Leave-States-Out RMSE: {np.mean(lso_rmse):.2f}\n")


# --- 2. Light Tuning Scored on 2021 Only ---
print("--- 2. Light Tuning Scored on 2021 ---")
param_grid = {
    "learning_rate": [0.01, 0.05, 0.1],
    "max_iter": [100, 300, 500],
    "min_samples_leaf": [5, 20, 50]
}

best_rmse = float('inf')
best_params = {}

for params in ParameterGrid(param_grid):
    # Train on 2017-2020
    m = HistGradientBoostingRegressor(**params, random_state=42)
    m.fit(train[best_features], train["Yield"])

    # Score strictly on the 2021 validation set[cite: 3]
    current_rmse = scores(val["Yield"], m.predict(val[best_features]))["RMSE"]

    if current_rmse < best_rmse:
        best_rmse = current_rmse
        best_params = params

print(f"Best 2021 Val RMSE: {best_rmse:.2f}")
print(f"Optimal Parameters: {best_params}\n")


# --- 3. Final Test on 2022 ---
print("--- 3. Final Test on 2022 ---")
# Combine 2017-2021 for the final training set
train_full = df[df["Year"].between(2017, 2021)]
test_2022  = df[df["Year"] == 2022].copy()

# Initialize the model with the best parameters found in Step 2
final_model = HistGradientBoostingRegressor(**best_params, random_state=42)
final_model.fit(train_full[best_features], train_full["Yield"])

# Generate final predictions and scores
test_2022["preds"] = final_model.predict(test_2022[best_features])
final_metrics = scores(test_2022["Yield"], test_2022["preds"])

print("Final 2022 Evaluation:")
for metric, value in final_metrics.items():
    print(f"{metric}: {value:.2f}")

--- 1. Leave-States-Out Validation ---
Average Leave-States-Out RMSE: 26.66

--- 2. Light Tuning Scored on 2021 ---
Best 2021 Val RMSE: 24.10
Optimal Parameters: {'learning_rate': 0.1, 'max_iter': 500, 'min_samples_leaf': 20}

--- 3. Final Test on 2022 ---
Final 2022 Evaluation:
RMSE: 24.90
MAE: 18.82
R2: 0.67
